# Proyecto Aprendizaje Estadístico
## Estimación del tiempo de entrega de pedidos de Food Delivery
**Sección 4.2 – Desarrollo e Implementación del Modelo** y **Sección 5.1 – Aplicación al Modelo**

Implementación en Google Colab (Python 3) con `python-weka-wrapper3`, que ejecuta los algoritmos de WEKA desde Python. Se emplean los archivos `train.arff` (40 000 registros) y `test.arff` (10 000 registros) generados en la etapa de preprocesamiento (sección 3).

### Paso 1. Instalación del entorno (Java + python-weka-wrapper3)

In [ ]:
!apt-get -qq update > /dev/null
!apt-get -qq install -y openjdk-11-jdk > /dev/null
!pip -q install python-weka-wrapper3
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"
print("Entorno listo")

### Paso 2. Carga de los data-sets de entrenamiento y prueba
Al ejecutar esta celda, seleccionar `train.arff` y `test.arff`.

In [ ]:
import os
from google.colab import files
if not (os.path.exists("train.arff") and os.path.exists("test.arff")):
    files.upload()
print([f for f in os.listdir(".") if f.endswith(".arff")])

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import weka.core.jvm as jvm
jvm.start(packages=False, max_heap_size="3g")

from weka.core.converters import Loader
from weka.classifiers import Classifier, Evaluation, FilteredClassifier
from weka.filters import Filter
from weka.core.classes import Random
from weka.core.version import weka_version
import weka.core.serialization as serialization

print("Versión de WEKA:", weka_version())
loader = Loader(classname="weka.core.converters.ArffLoader")
train = loader.load_file("train.arff"); train.class_is_last()
test  = loader.load_file("test.arff");  test.class_is_last()
print("Entrenamiento:", train.num_instances, "instancias,", train.num_attributes, "atributos")
print("Prueba:", test.num_instances, "instancias,", test.num_attributes, "atributos")
print("Variable objetivo:", train.class_attribute.name)

### Paso 3. Definición de los modelos

In [ ]:
def crear_modelos():
    # Regresión Lineal con Normalize encapsulado en FilteredClassifier (evita fuga de información)
    lr = FilteredClassifier()
    lr.filter = Filter(classname="weka.filters.unsupervised.attribute.Normalize", options=["-S", "1.0", "-T", "0.0"])
    lr.classifier = Classifier(classname="weka.classifiers.functions.LinearRegression", options=["-S", "0", "-R", "1.0E-8"])
    return {
        "ZeroR": Classifier(classname="weka.classifiers.rules.ZeroR"),
        "Regresión Lineal": lr,
        "M5P": Classifier(classname="weka.classifiers.trees.M5P", options=["-M", "4.0"]),
        "Random Forest": Classifier(classname="weka.classifiers.trees.RandomForest",
                                    options=["-I", "100", "-K", "0", "-M", "1.0", "-V", "0.001", "-S", "1", "-num-slots", "0"]),
        "Additive Regression": Classifier(classname="weka.classifiers.meta.AdditiveRegression",
                                          options=["-S", "0.1", "-I", "50", "-W", "weka.classifiers.trees.REPTree",
                                                   "--", "-M", "2", "-V", "0.001", "-N", "3", "-S", "1", "-L", "-1"]),
    }

for nombre, modelo in crear_modelos().items():
    print(f"{nombre}: {modelo.to_commandline()}")

### Paso 4. Validación cruzada de 5 pliegues sobre train.arff (semilla 42)
Tiempo aproximado: 4 a 5 minutos.

In [ ]:
filas = []
for nombre, modelo in crear_modelos().items():
    t0 = time.time()
    ev = Evaluation(train)
    ev.crossvalidate_model(modelo, train, 5, Random(42))
    filas.append({"Modelo": nombre,
                  "r": round(ev.correlation_coefficient, 4),
                  "MAE (min)": round(ev.mean_absolute_error, 3),
                  "RMSE (min)": round(ev.root_mean_squared_error, 3),
                  "RAE (%)": round(ev.relative_absolute_error, 2),
                  "RRSE (%)": round(ev.root_relative_squared_error, 2),
                  "Tiempo (s)": round(time.time() - t0, 1)})
    print(f"{nombre} listo")

tabla = pd.DataFrame(filas)
tabla

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
x = np.arange(len(tabla))
ax.bar(x - 0.2, tabla["MAE (min)"], 0.4, label="MAE")
ax.bar(x + 0.2, tabla["RMSE (min)"], 0.4, label="RMSE")
ax.set_xticks(x); ax.set_xticklabels(tabla["Modelo"], rotation=15)
ax.set_ylabel("Error (min)"); ax.set_title("Comparación de modelos – validación cruzada de 5 pliegues")
ax.legend(); ax.grid(axis="y", alpha=.3); plt.tight_layout(); plt.show()

### Paso 5. Importancia de variables (Random Forest) y selección del modelo

In [ ]:
rf = Classifier(classname="weka.classifiers.trees.RandomForest",
                options=["-I", "100", "-K", "0", "-M", "1.0", "-V", "0.001", "-S", "1", "-num-slots", "0", "-attribute-importance"])
rf.build_classifier(train)
texto = str(rf)
print(texto[texto.find("Attribute importance"):])

In [ ]:
candidatos = tabla[tabla["Modelo"] != "ZeroR"].sort_values(["MAE (min)", "RMSE (min)"])
mejor = candidatos.iloc[0]
print("Modelo seleccionado:", mejor["Modelo"])
print("MAE =", mejor["MAE (min)"], "min | RMSE =", mejor["RMSE (min)"], "min")
print("Reducción del error absoluto respecto a ZeroR:", round(100 - mejor["RAE (%)"], 1), "%")

---
## Sección 5.1 – Aplicación al Modelo: uso del data-set de entrenamiento y de prueba

In [ ]:
# Entrenamiento del modelo final con los 40 000 registros de train.arff
final = Classifier(classname="weka.classifiers.trees.M5P", options=["-M", "4.0"])
final.build_classifier(train)
serialization.write("m5p_final.model", final)
print(str(final).strip().split("\n")[-1])   # número de reglas
print("Modelo guardado como m5p_final.model")

In [ ]:
# Evaluación única sobre el conjunto de prueba (10 000 registros)
ev_test = Evaluation(train)
ev_test.test_model(final, test)
print(ev_test.summary("=== Evaluación final de M5P sobre test.arff ==="))

In [ ]:
# Análisis de las predicciones
iT = test.attribute_by_name("Traffic_Level").index
iW = test.attribute_by_name("Weather").index
pred = pd.DataFrame([{"Real": inst.get_value(test.class_index),
                      "Predicho": final.classify_instance(inst),
                      "Traffic_Level": inst.get_string_value(iT),
                      "Weather": inst.get_string_value(iW)} for inst in test])
pred["Error"] = pred["Predicho"] - pred["Real"]
pred["AbsError"] = pred["Error"].abs()

print("Error medio (sesgo):", round(pred["Error"].mean(), 3), "min")
for k in (2, 5, 10):
    print(f"% de predicciones con |error| <= {k} min:", round((pred["AbsError"] <= k).mean() * 100, 1))
print("Mayor error absoluto:", round(pred["AbsError"].max(), 2), "min")
print("\nMAE por nivel de tráfico:\n", pred.groupby("Traffic_Level")["AbsError"].agg(["count", "mean"]).round(2))
print("\nMAE por clima:\n", pred.groupby("Weather")["AbsError"].agg(["count", "mean"]).round(2))
trunc = pred["Real"] == 180
print("\nMAE registros truncados (=180):", round(pred.loc[trunc, "AbsError"].mean(), 2), "| resto:", round(pred.loc[~trunc, "AbsError"].mean(), 2))

recorte = pred["Predicho"].clip(10, 180) - pred["Real"]
print("\nPredicciones fuera de [10, 180]:", int(((pred["Predicho"] > 180) | (pred["Predicho"] < 10)).sum()))
print("Con recorte a [10, 180] -> MAE:", round(recorte.abs().mean(), 3), "| RMSE:", round(np.sqrt((recorte ** 2).mean()), 3))

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(pred["Real"], pred["Predicho"], s=4, alpha=.35)
ax.plot([0, 190], [0, 190], "r--", lw=1, label="Predicción perfecta (y = x)")
ax.set_xlabel("Tiempo real (min)"); ax.set_ylabel("Tiempo predicho por M5P (min)")
ax.set_title("M5P: valores reales vs. predichos en test.arff (n = 10 000)")
ax.legend(); ax.grid(alpha=.3); plt.tight_layout(); plt.show()

In [ ]:
# Aplicación del modelo guardado a tres pedidos nuevos (tiempo desconocido = ?)
cabecera = open("test.arff").read().split("@data")[0]
with open("nuevos.arff", "w") as f:
    f.write(cabecera + """@data
13,Wednesday,0,0,Clear,Commercial,Residential,Bike,5,4.5,4.3,Burger,2,Low,15,8,Low,4,Normal,?
20,Friday,0,0,Rain,CBD,Suburban,Scooter,3,4.0,4.1,Pizza,3,High,30,25,High,12,Normal,?
19,Saturday,1,1,Storm,Residential,Industrial,Bicycle,1,3.7,4.0,Biryani,5,High,35,40,Severe,15,Priority,?
""")
nuevos = loader.load_file("nuevos.arff"); nuevos.class_is_last()
modelo_cargado = Classifier(jobject=serialization.read("m5p_final.model"))
for n, inst in enumerate(nuevos, start=1):
    p = modelo_cargado.classify_instance(inst)
    print(f"Pedido {n}: {p:.1f} min  (recortado a [10, 180]: {min(max(p, 10), 180):.1f} min)")

---
## Sección VI – Evaluación
### 6.1 Métricas y evaluación del modelo

In [ ]:
# 6.1.a Conjunto completo de métricas de M5P sobre test.arff
y, yhat = pred["Real"].values, pred["Predicho"].values
err, ae = yhat - y, np.abs(yhat - y)
r2 = 1 - ((y - yhat) ** 2).sum() / ((y - y.mean()) ** 2).sum()
metricas = pd.DataFrame({
    "Métrica": ["Coeficiente de correlación (r)", "Coeficiente de determinación (R²)",
                "MAE (min)", "RMSE (min)", "Mediana del error absoluto (min)",
                "Percentil 90 del error absoluto (min)", "Percentil 95 del error absoluto (min)",
                "Error absoluto máximo (min)", "MAPE (%)", "Error medio / sesgo (min)",
                "RAE (%)", "RRSE (%)"],
    "Valor": [ev_test.correlation_coefficient, r2, ev_test.mean_absolute_error, ev_test.root_mean_squared_error,
              np.median(ae), np.percentile(ae, 90), np.percentile(ae, 95), ae.max(),
              (ae / y).mean() * 100, err.mean(), ev_test.relative_absolute_error, ev_test.root_relative_squared_error]})
metricas["Valor"] = metricas["Valor"].round(4)
metricas

In [ ]:
# 6.1.b Estabilidad de M5P en cada pliegue de la validación cruzada (mismo esquema: 5 pliegues, semilla 42)
from weka.core.dataset import Instances
datos = Instances.copy_instances(train)
rnd = Random(42)
datos.randomize(rnd)
pliegues = []
for i in range(5):
    tr_i = datos.train_cv(5, i, rnd)
    te_i = datos.test_cv(5, i)
    m = Classifier(classname="weka.classifiers.trees.M5P", options=["-M", "4.0"])
    m.build_classifier(tr_i)
    ev_i = Evaluation(tr_i); ev_i.test_model(m, te_i)
    pliegues.append({"Pliegue": i + 1, "Instancias de validación": te_i.num_instances,
                     "MAE (min)": round(ev_i.mean_absolute_error, 3), "RMSE (min)": round(ev_i.root_mean_squared_error, 3),
                     "r": round(ev_i.correlation_coefficient, 4)})
folds = pd.DataFrame(pliegues)
print(folds.to_string(index=False))
print(f"\nMAE  = {folds['MAE (min)'].mean():.3f} ± {folds['MAE (min)'].std():.3f} min")
print(f"RMSE = {folds['RMSE (min)'].mean():.3f} ± {folds['RMSE (min)'].std():.3f} min")

In [ ]:
# 6.1.c Control de sobreajuste: error de entrenamiento vs. validación cruzada vs. prueba
ev_train = Evaluation(train); ev_train.test_model(final, train)
sobreajuste = pd.DataFrame({
    "Conjunto": ["Entrenamiento (40 000)", "Validación cruzada (5 pliegues)", "Prueba (10 000)"],
    "MAE (min)": [round(ev_train.mean_absolute_error, 3), round(folds["MAE (min)"].mean(), 3), round(ev_test.mean_absolute_error, 3)],
    "RMSE (min)": [round(ev_train.root_mean_squared_error, 3), round(folds["RMSE (min)"].mean(), 3), round(ev_test.root_mean_squared_error, 3)]})
sobreajuste

In [ ]:
# 6.1.d Distribución de los residuos (predicho − real)
fig, axs = plt.subplots(1, 2, figsize=(12, 4.5))
axs[0].hist(err, bins=60, color="#1f5fa8", alpha=.8)
axs[0].axvline(0, color="r", ls="--"); axs[0].set_title("Distribución de residuos (test.arff)")
axs[0].set_xlabel("Residuo: predicho − real (min)"); axs[0].set_ylabel("Frecuencia")
axs[1].scatter(yhat, err, s=4, alpha=.3, color="#1f5fa8")
axs[1].axhline(0, color="r", ls="--"); axs[1].set_title("Residuos vs. valor predicho")
axs[1].set_xlabel("Tiempo predicho (min)"); axs[1].set_ylabel("Residuo (min)")
for a in axs: a.grid(alpha=.3)
plt.tight_layout(); plt.show()

### 6.2 Evaluación de desempeño del Sistema de IA

In [ ]:
# 6.2.a Comparación con el método tradicional: preparación + distancia / velocidad promedio
import io
def arff_a_df(ruta):
    cab, dat = open(ruta).read().split("@data")
    cols = [l.split()[1] for l in cab.splitlines() if l.lower().startswith("@attribute")]
    return pd.read_csv(io.StringIO(dat.strip()), header=None, names=cols, quotechar="'", skipinitialspace=True)
tr_df, te_df = arff_a_df("train.arff"), arff_a_df("test.arff")

# Velocidades estimadas SOLO con train.arff (registros no truncados)
ok = (tr_df.Time_taken_min < 180) & (tr_df.Time_taken_min > tr_df.Preparation_Time_Min)
tr_ok = tr_df[ok].assign(Recorrido_h=lambda d: (d.Time_taken_min - d.Preparation_Time_Min) / 60)
v_global = tr_ok.Road_Distance_km.sum() / tr_ok.Recorrido_h.sum()
v_seg = (tr_ok.groupby(["Vehicle_Type", "Traffic_Level"]).Road_Distance_km.sum()
         / tr_ok.groupby(["Vehicle_Type", "Traffic_Level"]).Recorrido_h.sum()).rename("v")
print(f"Velocidad promedio global: {v_global:.2f} km/h")

eta_global = (te_df.Preparation_Time_Min + te_df.Road_Distance_km / v_global * 60).clip(10, 180)
v_te = te_df.join(v_seg, on=["Vehicle_Type", "Traffic_Level"])["v"]
eta_seg = (te_df.Preparation_Time_Min + te_df.Road_Distance_km / v_te * 60).clip(10, 180)

def resumen(nombre, p):
    e = p - te_df.Time_taken_min.values
    return {"Método": nombre, "MAE (min)": round(np.abs(e).mean(), 3), "RMSE (min)": round(np.sqrt((e ** 2).mean()), 3),
            "% |error| ≤ 5 min": round((np.abs(e) <= 5).mean() * 100, 1),
            "% subestimado > 5 min": round((e < -5).mean() * 100, 1),
            "% subestimado > 10 min": round((e < -10).mean() * 100, 1)}
comparacion = pd.DataFrame([resumen("Tradicional: velocidad promedio global", eta_global.values),
                            resumen("Tradicional: velocidad por vehículo y tráfico", eta_seg.values),
                            resumen("Sistema de IA (M5P)", yhat)])
comparacion

In [ ]:
# 6.2.b Desempeño computacional
t0 = time.time()
tmp = Classifier(classname="weka.classifiers.trees.M5P", options=["-M", "4.0"]); tmp.build_classifier(train)
t_entrenamiento = time.time() - t0
t0 = time.time()
for inst in test:
    final.classify_instance(inst)
t_pred_ms = (time.time() - t0) / test.num_instances * 1000
print(f"Tiempo de entrenamiento (40 000 registros): {t_entrenamiento:.1f} s")
print(f"Tiempo promedio de predicción por pedido: {t_pred_ms:.3f} ms")
print(f"Tamaño del modelo guardado: {os.path.getsize('m5p_final.model') / 1024:.0f} KB")

### 6.2.c Sistema de estimación del tiempo de entrega
Modifique los datos del pedido en el formulario y ejecute la celda para obtener el tiempo estimado.

In [ ]:
#@title Estimador de tiempo de entrega (M5P) { run: "auto" }
Dia = "Friday" #@param ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
Hora = 20 #@param {type:"slider", min:0, max:23, step:1}
Festivo = False #@param {type:"boolean"}
Clima = "Rain" #@param ["Clear", "Cloudy", "Fog", "Rain", "Storm"]
Trafico = "High" #@param ["Low", "Moderate", "High", "Severe"]
Zona_recojo = "CBD" #@param ["CBD", "Commercial", "Industrial", "Residential", "Suburban"]
Zona_entrega = "Suburban" #@param ["CBD", "Commercial", "Industrial", "Residential", "Suburban"]
Vehiculo = "Scooter" #@param ["Bicycle", "Bike", "Electric Scooter", "Scooter"]
Experiencia_repartidor_anios = 3 #@param {type:"slider", min:0.1, max:15, step:0.1}
Calificacion_repartidor = 4.0 #@param {type:"slider", min:3.2, max:5, step:0.1}
Calificacion_restaurante = 4.1 #@param {type:"slider", min:3.3, max:5, step:0.1}
Tipo_cocina = "Pizza" #@param ["Bakery", "Biryani", "Burger", "Cafe", "Chinese", "Desserts", "North Indian", "Pizza", "South Indian"]
Cantidad_items = 3 #@param {type:"slider", min:1, max:7, step:1}
Carga_restaurante = "High" #@param ["Low", "Medium", "High"]
Tiempo_preparacion_min = 30 #@param {type:"slider", min:5, max:51, step:1}
Distancia_km = 25.0 #@param {type:"number"}
Numero_semaforos = 12 #@param {type:"slider", min:2, max:18, step:1}
Prioridad = "Normal" #@param ["Normal", "Priority", "VIP"]

fin_semana = 1 if Dia in ("Saturday", "Sunday") else 0
fila = [Hora, f"'{Dia}'", fin_semana, int(Festivo), f"'{Clima}'", f"'{Zona_recojo}'", f"'{Zona_entrega}'",
        f"'{Vehiculo}'", Experiencia_repartidor_anios, Calificacion_repartidor, Calificacion_restaurante,
        f"'{Tipo_cocina}'", Cantidad_items, f"'{Carga_restaurante}'", Tiempo_preparacion_min, Distancia_km,
        f"'{Trafico}'", Numero_semaforos, f"'{Prioridad}'", "?"]
with open("pedido.arff", "w") as f:
    f.write(open("test.arff").read().split("@data")[0] + "@data\n" + ",".join(map(str, fila)) + "\n")
pedido = loader.load_file("pedido.arff"); pedido.class_is_last()
estimado = Classifier(jobject=serialization.read("m5p_final.model")).classify_instance(pedido.get_instance(0))
estimado_rec = min(max(estimado, 10), 180)

print("=" * 52)
print("  SISTEMA DE ESTIMACIÓN DEL TIEMPO DE ENTREGA (M5P)")
print("=" * 52)
print(f"  Tiempo estimado de entrega: {estimado_rec:.0f} minutos")
print(f"  Rango orientativo (± MAE de prueba): {max(estimado_rec - 2.3, 10):.0f} – {min(estimado_rec + 2.3, 180):.0f} min")
if estimado > 180:
    print("  Aviso: la estimación supera el máximo registrado; interpretar como 180 minutos o más.")
if not (0.08 <= Distancia_km <= 70.82):
    print("  Aviso: la distancia está fuera del rango de entrenamiento (0.08 – 70.82 km).")
print("=" * 52)